# OTTO · Round 04 — rolling demand versus frozen demand

**The index notebook must finish first.** The same 75 formulas are evaluated from two snapshot policies. Existing 134-column controls and candidate pools are reused. The primary is rolling demand versus control; the frozen-snapshot ablation cannot rescue a failed primary. Up to twelve new experiment models.

In [1]:
from pathlib import Path
import importlib.util
import json
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'demand_features.py').is_file() and (p / 'launch.py').is_file()),
            Path.home() / 'otto_feature_round04')
if not (ROOT / 'demand_features.py').is_file():
    raise RuntimeError('Open this notebook from the extracted otto_feature_round04 folder')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location('otto_round04_launcher', ROOT / 'launch.py')
launch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launch)
sys.modules['launch'] = launch
state = {'halted': False, 'completed': []}
ML = Path.home() / 'otto-recommender-system/.venv/bin/python'
print('KERNEL_READY')
print('Notebook Python:', sys.executable)
print('ML Python:', ML)

def stage(name):
    if state['halted']:
        raise RuntimeError('Earlier stage stopped. Do not continue; collect the return ZIP.')
    try:
        value = launch.run_stage(name)
    except BaseException:
        state['halted'] = True
        raise
    state['completed'].append(name)
    return value


KERNEL_READY
Notebook Python: /opt/conda/bin/python
ML Python: /home/sagemaker-user/otto-recommender-system/.venv/bin/python


## Verify index gate — no automatic rebuild

In [2]:
s = json.loads((ROOT/'outputs/index_summary.json').read_text())
assert s['status'] == 'ROUND04_INDEX_READY'
assert s['excluded_study_sessions'] == 5120
print('INDEX_GATE_PASSED')

INDEX_GATE_PASSED


## Build only the two demand representations
75 columns from an hourly snapshot and the same 75 from fixed Aug16 22:00 UTC history. All 400 candidates retained before negative sampling. 64-session checkpoints; first 16 sessions replayed numerically.

In [3]:
stage('features')

RUNNING features; process cap 320s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round04/outputs/features.log
{"completed": 256, "event": "feature_checkpoints", "stage": "rolling_demand_features", "total": 4096, "utc": "2026-09-12T02:42:00.271861+00:00"}
{"completed": 512, "event": "feature_checkpoints", "stage": "rolling_demand_features", "total": 4096, "utc": "2026-09-12T02:42:03.832369+00:00"}
{"completed": 768, "event": "feature_checkpoints", "stage": "rolling_demand_features", "total": 4096, "utc": "2026-09-12T02:42:07.387005+00:00"}
2026-09-12T02:42:10.214279+00:00 LAUNCHER_HEARTBEAT phase=features seconds=15.0
{"completed": 960, "elapsed_seconds": 15.0, "event": "heartbeat", "stage": "rolling_demand_features", "total": 4096, "utc": "2026-09-12T02:42:10.285373+00:00"}
{"completed": 1024, "event": "feature_checkpoints", "stage": "rolling_demand_features", "total": 4096, "utc": "2026-09-12T02:42:10.929576+00:00"}
{"completed": 1280, "event": "feature_checkpoints"

{'phase': 'features', 'exit_code': 0}

## Matched predictive screen
All six saved controls must replay correctly before the first new challenger fit. Same training rows, folds, model capacity, labels censored at validation cutoff, and complete validation denominators. Twelve new experiment fits maximum; no model search.

In [4]:
stage('screen')

RUNNING screen; process cap 260s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round04/outputs/screen.log
2026-09-12T02:43:08.566023+00:00 LAUNCHER_HEARTBEAT phase=screen seconds=15.0
{"completed": 3, "elapsed_seconds": 15.0, "event": "heartbeat", "stage": "verify_all_saved_controls", "total": 6, "utc": "2026-09-12T02:43:08.636927+00:00"}
{"event": "all_six_controls_verified_before_new_fits", "utc": "2026-09-12T02:43:12.076697+00:00"}
{"arm": "demand_rolling", "completed": 1, "event": "model_checkpoint", "fold": 0, "new_fit": 1, "objective": "clicks", "stage": "matched_rolling_demand", "total": 12, "utc": "2026-09-12T02:43:17.093654+00:00"}
{"arm": "demand_rolling", "completed": 2, "event": "model_checkpoint", "fold": 0, "new_fit": 1, "objective": "carts", "stage": "matched_rolling_demand", "total": 12, "utc": "2026-09-12T02:43:22.442196+00:00"}
2026-09-12T02:43:23.584912+00:00 LAUNCHER_HEARTBEAT phase=screen seconds=30.0
{"completed": 2, "elapsed_seconds": 30.0, "ev

{'phase': 'screen', 'exit_code': 0}

## Audit saved integers and create seven Plotly charts
This reads saved per-session hit statistics. It does not fit models.

In [5]:
stage('report')

RUNNING report; process cap 60s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round04/outputs/report.log
RESULT: ROUND04_REPORT_READY
FINISHED report: 1.2 seconds
RETURN_FILE: /home/sagemaker-user/otto_feature_round04/otto_round04_return.zip


{'phase': 'report', 'exit_code': 0}

## Save and collect
Use 03_saved_results.ipynb for repeated viewing, not this execution notebook. Stop the SageMaker application after downloading the ZIP.

In [6]:
if state['halted']:
    raise RuntimeError('Earlier stage did not complete')
print('ROUND04_EXECUTION_COMPLETE')
launch.collect()

ROUND04_EXECUTION_COMPLETE
RETURN_FILE: /home/sagemaker-user/otto_feature_round04/otto_round04_return.zip


'/home/sagemaker-user/otto_feature_round04/otto_round04_return.zip'